# Task B: System Identification \(5P\)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import scipy as sp
import pinocchio as pin

You are given the task to identify the inertial parameters of the Unitree Go2 leg.
Your extremely competent colleague has conducted an experiment with the front left leg and sent you the data in **go2_leg_exp_500hz.csv** and **go2_leg_exp_1khz.csv** (Both are from the same experiement at different sampling rates). They have the following structure:
| column number(s) | column name(s) |description | unit |
| --- | --- | --- | --- |
| 0 | time | timestamp | s |
| 1-3 | q0-q2| measured joint positions | rad |
| 4-6 | dq0-dq2| desired joint velocities | rad/s |
| 7-9 | tau0-tau2| measured joint torques | N.m |

He has also made a video of the experiment's last 45 seconds. Run the following cell to see it.

In [ ]:
from IPython.display import Video
Video("media/go2_leg_exp.mp4",embed=True, width=640, height=480)

Let's take a look at the data:

We will read the *.csv* file using the [pandas library](https://pandas.pydata.org/docs/). Lets plot the measured joint data first.

In [ ]:
df = pd.read_csv('data/go2_leg_exp_500hz.csv', sep=',') #available datasets sampled at 1khz:go2_leg_exp_1khz, or downsampled to 500hz: go2_leg_exp_500hz

joint_no = 0 # choose which joint you would like to plot the data of. 0, 1, 2

# normalize time column
df["time"] = (df["time"] - df["time"][0])

# extract arrays from dataframe
time = df["time"].values
tau = df[f"tau{joint_no}"].values
q = df[f"q{joint_no}"].values
dq = df[f"dq{joint_no}"].values

# plot
fig, ax = plt.subplots(3,1,sharex=True, figsize=(10,6))
ax[0].plot(time, tau, color='blue')
ax[0].set_ylabel('tau (N.m)')
ax[0].grid(axis='y')

ax[1].plot(time, q, color='red')
ax[1].set_ylabel('q (rad)')
ax[1].grid(axis='y')

ax[2].plot(time, dq, color='green')
ax[2].set_xlabel('time (s)')
ax[2].set_ylabel('dq (rad/s)')
ax[2].grid(axis='y')

fig.suptitle(f'Measured Data of Joint {joint_no}')
fig.tight_layout()

**(a)** Prepare the data for the identification. Use appropriate filters, compute missing data, and reject outliers if necessary.
*Hint: Use the [scipy library](https://docs.scipy.org/doc/scipy/) for this task. You can append the filtered and computed signals to the dataframe. It is a good idea to name the new columns to avoid confusion*

In [ ]:
# filter existing data
#TODO
# compute missing data
#TODO
# reject outliers
#TODO

Lets have a look at your results. We will plot them together with the measured data from above. The variables for plotting the measured data are taken from the second code block of this notebook.

In [ ]:
# extract arrays
tau_filt = #TODO
q_filt = #TODO
dq_filt = #TODO
missing = #TODO

# plot
fig, ax = plt.subplots(4,1,sharex=True, figsize=(10,8))
ax[0].plot(time, tau, color='lightskyblue', label='measured tau')
ax[0].plot(time, tau_filt, color='blue', label='filtered tau')
ax[0].set_ylabel('tau (N.m)')
ax[0].grid(axis='y')
ax[0].legend()

ax[1].plot(time, q, color='lightcoral', label ='measured q')
ax[1].plot(time, q_filt, color='red', label='filtered q')
ax[1].set_ylabel('q (rad)',)
ax[1].grid(axis='y')
ax[1].legend()

ax[2].plot(time, dq, color='lightgreen', label='measured dq')
ax[2].plot(time, dq_filt, color='green', label='filtered dq')
ax[2].set_ylabel('dq (rad/s)')
ax[2].grid(axis='y')
ax[2].legend()

ax[3].plot(time, missing, color='orange', label='measured missing')
ax[3].set_xlabel('time (s)')
ax[3].set_ylabel('missing (?)')
ax[3].grid(axis='y')
ax[3].legend()
fig.suptitle(f'Measured and Processed Data of Joint {joint_no}')
fig.tight_layout()


**(b)** Explain your choice of methods and discuss their advantages and disadvantages.

**(c)** Obtain the regressor matrix using the pinocchio library. Note that the full matrix is constructed by vertically stacking the regressors from each time step. Additionally you need to extract the joint torque vector for each time step from the dataframe and stack them vertically.
*Hint: use the functions: [pin.computeJointTorqueRegressor](https://gepettoweb.laas.fr/doc/stack-of-tasks/pinocchio/devel/doxygen-html/namespacepinocchio.html#a9aa8452f0a804c5ed38c3b3f03fdd2eb), [np.vstack](https://numpy.org/doc/2.1/reference/generated/numpy.vstack.html) and [np.concatenate](https://numpy.org/doc/stable/reference/generated/numpy.concatenate.html).
Afterwards estimate all model parameters using a **least squares regression** with the help of the **pseudo inverse**.

In [ ]:
#### DO NOT MODIFY THIS CODE BLOCK ####
urdf_file = "urdf/go2_leg_description.urdf"
model = pin.buildModelFromUrdf(urdf_file)
data = model.createData()
print(model.name) #check if model is loaded it should print go2_description

In [ ]:
regressor = []
tau_vec = []

#TODO Hint: you can loop over the rows of the dataframe

#check dimensions
assert regressor.shape == (3 * df.shape[0], 30), 'The dimensions of the regressor matrix are not correct'

# estimate parameters
parameter_vector = #TODO

**(d)** Validate your results. For that **print** the parameters you computed alongside the parameters from the URDF. Furthermore **compute the joint torques that your new model predicts** and compare them with the measured ones by ...
1. ...computing the torque **root mean squared error (RMSE)**.
2. ...**plotting** them together.

In [ ]:
#compute predicted torques
tau_predicted = #TODO

fig= plt.figure(figsize=(10,8))
colors = [
    'lightskyblue',
    'blue',
    ]
link_idx = 2
# compare computed parameters with URDF
print(f'Identified Parameters of link {link_idx}: {parameter_vector[link_idx*10:link_idx*10 + 10]}')
print(f'Parameters of link {link_idx} from URDF: {model.inertias[link_idx].toDynamicParameters()}')
#compute torque RMSE for each joint
tau_pred = tau_predicted[link_idx::3]
tau_measured = df[f"tau_filt{link_idx}"].to_numpy()

rmse = #TODO
print(f'RMSE of joint torque prediction of joint {link_idx}: {rmse}')

print('\n')
# plot
plt.plot(time, tau_measured, label=f'Measured torques of joint {link_idx}', color=colors[0])
plt.plot(time, tau_pred, label=f'Predicted torques of joint {link_idx}', color=colors[1])

plt.xlabel('Time (s)')
plt.ylabel('Torque (Nm)')
plt.title(f'Torque Comparison for Joint {link_idx}')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

Discuss your results by answering to the following questions:

1. How close are your identified parameters to the ones from the URDF? Explain possible differences.
2. How well do the identified parameters predict the joint torques for the whole dataset? How could you check if they generalize well to new data?
3. How can the classic least squares identification be improved? 

**ANSWER HERE**